#My temporary idea: pipeline overview

1. Data understanding, EDA
2. Test relationships and construct relationshp graph
3. Features engenieering, turn graph into ML input
4. ML model
5. Cross wafer validation
6. Evaluation and interpretation

First we're loading up the data and organizing the files in 4 csv:
- 1 csv with the threshold (only two rows)
- 3 csv, one for each wafer

In [ ]:
import pandas as pd


print(" STEP 1 — EDA STARTING ")

# 0. LOAD DATA
print("Loading dataset...")

# Method 1: Direct full path
df = pd.read_csv("/Users/geotech/Desktop/Master_AI/Master/AML/Data_Analysis/data.csv")

# Method 2: Using raw string (useful for Windows paths)
# df = pd.read_csv(r"/Users/geotech/Desktop/Master_AI/Master/AML/Data_Analysis/data.csv")


print(f"Dataset shape: {df.shape}")
print("First rows preview:")
print(df.head(), "\n")

In [ ]:
print("All column names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:3}. {col}")

# Also show count
print(f"Total columns: {len(df.columns)}")

In [ ]:
# 1. IDENTIFY WAFERS

print("Splitting by wafer...")

if "Wafer" in df.columns:
    wafers = df["Wafer"].unique()
    wafer_groups = {w: df[df["Wafer"] == w].copy() for w in wafers}
else:
    raise ValueError("No 'Wafer' column found. Please check dataset structure.")

print(f"Found wafers: {list(wafers)}")

In [ ]:
# 1. EXTRACT THRESHOLD LINES
print("Extracting threshold lines...")

# Extract first 2 lines as thresholds
threshold_lines = df.iloc[:2].copy()
threshold_lines.to_csv("thresholds.csv", index=False)
print(f"Saved first 2 lines (thresholds) to 'thresholds.csv'")
print(f"Threshold lines shape: {threshold_lines.shape}\n")

# Remove threshold lines from main dataframe
df_data = df.iloc[2:].reset_index(drop=True)
print(f"Data after removing thresholds: {df_data.shape}")


In [ ]:
# SIMPLIFIED EXTRACTION & GROUPING

print("Extracting and grouping by wafer number...")

# Extract wafer number using simple string slicing
# The pattern is always 'DMKYXXX-' where XXX is the wafer number
df_data['Wafer_Number'] = df_data['Wafer'].str.extract(r'DMKY(\d{3})-')

# Remove rows without valid wafer number (should be none if all follow pattern)
df_data_clean = df_data.dropna(subset=['Wafer_Number']).copy()

# Group by wafer number and save in one go
for wafer_num, group in df_data_clean.groupby('Wafer_Number'):
    filename = f"wafer_{wafer_num}.csv"
    group.drop('Wafer_Number', axis=1).to_csv(filename, index=False)
    print(f"  ✓ Wafer {wafer_num}: {len(group)} rows → '{filename}'")

print(f"\n✓ Created files for wafers: {df_data_clean['Wafer_Number'].unique().tolist()}")

In [ ]:
# Check shapes of all wafer CSV files
for wafer_num in ['801', '806', '812']:
    filename = f"wafer_{wafer_num}.csv"
    try:
        df = pd.read_csv(filename)
        print(f"{filename}: {df.shape[0]} rows, {df.shape[1]} columns")
    except FileNotFoundError:
        print(f"{filename}: NOT FOUND")

Since I dont wanna re-make the files everytime and i already have them saved from doing this locally, i will just load up the csv i have stored on my device

In [ ]:
print(" STEP 1 — LOADING EXISTING FILES ")

# Define the folder path
folder_path = "/Users/geotech/Desktop/Master_AI/Master/AML/Data_Analysis"

# Load threshold file
thresholds = pd.read_csv(f"{folder_path}/thresholds.csv")
print(f"✓ Loaded thresholds.csv: {thresholds.shape[0]} rows, {thresholds.shape[1]} columns")

# Load the 3 wafer files
wafer_801 = pd.read_csv(f"{folder_path}/wafer_801.csv")
wafer_806 = pd.read_csv(f"{folder_path}/wafer_806.csv")
wafer_812 = pd.read_csv(f"{folder_path}/wafer_812.csv")

print(f"✓ Loaded wafer_801.csv: {wafer_801.shape[0]} rows, {wafer_801.shape[1]} columns")
print(f"✓ Loaded wafer_806.csv: {wafer_806.shape[0]} rows, {wafer_806.shape[1]} columns")
print(f"✓ Loaded wafer_812.csv: {wafer_812.shape[0]} rows, {wafer_812.shape[1]} columns")

# Optional: Preview first few rows
print("\nThresholds preview:")
print(thresholds.head(), "\n")

print("Wafer 801 preview:")
print(wafer_801.head())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import shutil

# Create output folder
output_dir = "correlation_results"
os.makedirs(output_dir, exist_ok=True)

# --- CONFIG ---
SAVE_RESULTS = True          # master switch
MIN_FREE_GB = 0.5            # minimum free space required

def has_enough_space(path, min_gb):
    total, used, free = shutil.disk_usage(path)
    return free / (1024**3) > min_gb

def clean_numeric(df):
    df = df.apply(pd.to_numeric, errors='coerce')

    # drop columns with too many NaNs
    df = df.loc[:, df.isna().mean() < 0.2]

    # drop constant columns (important for correlation stability)
    df = df.loc[:, df.nunique(dropna=True) > 1]

    return df

def plot_corr(corr, wafer_num):
    plt.figure(figsize=(10, 8))
    im = plt.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
    plt.colorbar(im)

    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=6)
    plt.yticks(range(len(corr.index)), corr.index, fontsize=6)

    plt.title(f"Correlation Matrix - Wafer {wafer_num}")
    plt.tight_layout()
    plt.show()

    return im


# Process wafers
for wafer_num, wafer_df in [('801', wafer_801), ('806', wafer_806), ('812', wafer_812)]:
    print(f"\nProcessing Wafer {wafer_num}...")

    meta_cols = [
        "Source Lot", "Lot", "Wafer", "rework_flag", "Program", "temperature",
        "subid", "site", "die_x", "die_y", "device_nr", "rom_code",
        "hardbin", "lib_info", "BinName", "BinState"
    ]
    meta_cols = [c for c in meta_cols if c in wafer_df.columns]
    test_cols = [c for c in wafer_df.columns if c not in meta_cols]

    df_tests = clean_numeric(wafer_df[test_cols])

    if df_tests.shape[1] < 2:
        print(f"  ⚠ Not enough valid test columns for wafer {wafer_num}")
        continue

    corr_matrix = df_tests.corr(method='pearson')

    # ---- MODE DECISION ----
    can_save = SAVE_RESULTS and has_enough_space(output_dir, MIN_FREE_GB)

    if can_save:
        csv_path = f"{output_dir}/correlation_wafer_{wafer_num}.csv"
        img_path = f"{output_dir}/correlation_wafer_{wafer_num}.png"

        corr_matrix.to_csv(csv_path)

        plt.figure(figsize=(10, 8))
        plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
        plt.colorbar()
        plt.title(f"Correlation Matrix - Wafer {wafer_num}")
        plt.savefig(img_path, dpi=120, bbox_inches='tight')
        plt.close()

        print(f"  ✓ Saved CSV + PNG for wafer {wafer_num}")

    else:
        print(f"  ⚠ Not enough disk space OR saving disabled → showing summary")

        # summary output instead of saving
        print("\nTop correlations (absolute):")

        corr_abs = corr_matrix.abs().unstack()
        corr_abs = corr_abs[corr_abs < 1].sort_values(ascending=False).head(10)

        print(corr_abs)

        # safer plotting (works better in notebooks than imshow alone)
        plot_corr(corr_matrix, wafer_num)

print("\n✓ Done processing all wafers.")

Now we compute the convergence matrix: Think of each test as a “sensor” watching thousands of chips: the correlation matrix is just checking whether two sensors wiggle together across all those chips—if whenever test A is a bit high on a chip, test B is also a bit high, they get a strong positive correlation (close to 1); if one goes up while the other goes down, it’s negative; and if they behave independently, it’s near zero—so you’re not measuring failure yet, just whether tests behave like copies, cousins, or strangers across the same population.